# 3.12 MoE 规模化训练深挖

> 🕐 预估学习时间：45分钟

稀疏 MoE 的难点不在“多几个 FFN”，而在路由坍塌、专家负载不均、EP 通信与容量因子。本节深挖工业训练技巧。

深挖点：
- 辅助损失 vs 无辅助损失偏置
- capacity factor 与 token drop
- Expert Parallel 通信量
- 共享专家 + 细粒度专家


## 1. 路由坍塌演示

无约束时，路由器可能把几乎所有 token 送给少数专家。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


class Router(nn.Module):
    def __init__(self, d=32, n_experts=8):
        super().__init__()
        self.gate = nn.Linear(d, n_experts, bias=False)
        self.n_experts = n_experts

    def forward(self, x, bias=None):
        logits = self.gate(x)
        if bias is not None:
            logits = logits + bias
        return torch.softmax(logits, dim=-1)


def usage_entropy(probs):
    u = probs.mean(0)
    return float(-(u * (u + 1e-12).log()).sum()), u


x = torch.randn(2000, 32)
# pathological init: large first row
router = Router()
with torch.no_grad():
    router.gate.weight.zero_()
    router.gate.weight[0] += 5

probs = router(x)
ent, usage = usage_entropy(probs.detach())
print('=== Collapse without balancing ===')
print('usage', usage.tolist())
print(f'entropy={ent:.3f} (max={torch.log(torch.tensor(8.0)):.3f})')
print('Key: Collapsed routing wastes capacity and hurts quality.')



## 2. 负载均衡：辅助损失与偏置反馈

- **Switch/GShard 辅助损失**：`N · Σ f_i · P_i`  
- **无辅助损失（DeepSeek 风格直觉）**：对专家加可学习/反馈偏置，压低过热专家


In [ ]:
def aux_load_balance(probs, topk_idx, n_experts):
    # f: fraction of tokens assigned; P: mean router prob
    N = n_experts
    f = torch.zeros(N)
    for i in topk_idx.view(-1):
        f[i] += 1
    f = f / f.sum()
    P = probs.mean(0)
    return N * (f * P).sum(), f, P


def train_with_balance(use_bias_feedback=False, steps=60):
    r = Router(n_experts=8)
    experts = nn.ModuleList([nn.Linear(32, 32) for _ in range(8)])
    opt = torch.optim.Adam(list(r.parameters()) + list(experts.parameters()), lr=1e-2)
    bias = torch.zeros(8)
    hist = []
    for step in range(steps):
        x = torch.randn(256, 32)
        probs = r(x, bias if use_bias_feedback else None)
        topv, topi = probs.topk(2, dim=-1)
        # dispatch weighted expert outs
        out = 0
        for k in range(2):
            w = topv[:, k:k+1]
            # gather expert outputs (loop for clarity)
            eo = torch.stack([experts[i](x[t:t+1]) for t, i in enumerate(topi[:, k])], 0).squeeze(1)
            out = out + w * eo
        task = (out - x).pow(2).mean()  # reconstruct
        aux, f, P = aux_load_balance(probs, topi, 8)
        loss = task + (0.0 if use_bias_feedback else 0.01 * aux)
        opt.zero_grad(); loss.backward(); opt.step()
        if use_bias_feedback:
            # push down hot experts
            with torch.no_grad():
                bias -= 0.1 * (f - 1.0 / 8)
        if step % 20 == 0 or step == steps - 1:
            ent, usage = usage_entropy(probs.detach())
            hist.append((step, ent, usage.clone()))
            print(f'step={step} mode={"bias" if use_bias_feedback else "aux"} ent={ent:.3f} usage={usage.tolist()}')
    return hist


print('=== Aux-loss balancing ===')
train_with_balance(False)
print('=== Bias-feedback balancing ===')
train_with_balance(True)
print('Key: Either penalize imbalance or continuously nudge expert biases.')


## 3. Capacity factor 与 drop

每专家最多处理 `capacity = CF · tokens · topk / experts`。超出则 drop 或改道。CF 太小伤质量，太大伤效率。


In [ ]:
def capacity_and_drops(n_tokens=1024, n_experts=8, topk=2, cf=1.25):
    cap = int(cf * n_tokens * topk / n_experts)
    # simulate random assignments
    assign = torch.randint(0, n_experts, (n_tokens, topk))
    loads = torch.zeros(n_experts)
    drops = 0
    for t in range(n_tokens):
        for k in range(topk):
            e = int(assign[t, k])
            if loads[e] < cap:
                loads[e] += 1
            else:
                drops += 1
    return cap, loads, drops / (n_tokens * topk)


print('=== Capacity Factor Sweep ===')
for cf in [1.0, 1.25, 2.0]:
    cap, loads, drop_rate = capacity_and_drops(cf=cf)
    print(f'CF={cf:.2f} cap={cap} drop_rate={drop_rate:.3f} load_std={loads.std().item():.2f}')
print('Key: Tune CF on drop_rate vs MFU; monitor per-expert occupancy histograms.')


## 4. Expert Parallel 通信

Token 按专家 ID all-to-all 到专家所在 rank，算完再返回。通信 ∝ 被路由激活量，不是全参数量。


In [ ]:
def ep_comm_bytes(n_tokens, hidden, topk=2, dtype=2, P=8):
    # each token sends topk hidden states out and back
    return 2 * n_tokens * topk * hidden * dtype * (P - 1) / P


print('=== EP Communication ===')
for P in [8, 16, 64]:
    b = ep_comm_bytes(8192, 2048, P=P)
    print(f'EP={P}: ~{b/1024**3:.3f} GB/step (toy)')
print('Key: MoE scales compute faster than dense, but EP all-to-all becomes the new bottleneck.')


## 课后思考题

1. 细粒度专家（更多更小）如何改变 CF 与通信形态？
2. 共享专家解决了什么冗余问题？代价是什么？
3. 推理时 expert packing / 动态batch 如何避免气泡？
4. 路由日志如何用于数据诊断（某域总进某专家）？

---
> 本节是MoE 规模化训练的垂直深挖。建议对照真实训练日志/线上指标复现关键实验，而不是只跑通玩具代码。
